In [1]:
import requests
import pandas as pd

# Define the API endpoint
url = 'https://jsonplaceholder.typicode.com/posts'

# Fetch 100 posts
response = requests.get(url)
posts = response.json()

# Convert to Pandas DataFrame
df = pd.DataFrame(posts)

# Display the first 5 rows and info to understand the structure
print("Original DataFrame head:")
display(df.head())
print("\nOriginal DataFrame info:")
df.info()

Original DataFrame head:


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...



Original DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   userId  100 non-null    int64 
 1   id      100 non-null    int64 
 2   title   100 non-null    object
 3   body    100 non-null    object
dtypes: int64(2), object(2)
memory usage: 3.3+ KB


In [2]:
data_quality_report = {
    'before_cleaning': {
        'row_count': len(df),
        'issues': {
            'null_counts': {},
            'duplicate_rows': 0,
            'type_mismatches': {},
            'out_of_range_values': {},
            'inconsistent_string_formats': {}
        }
    },
    'after_cleaning': {
        'row_count': 0,
        'issues_fixed': {
            'null_values_fixed': 0,
            'duplicate_rows_fixed': 0,
            'type_mismatches_fixed': 0,
            'out_of_range_values_fixed': 0,
            'inconsistent_string_formats_fixed': 0
        }
    }
}

# 1. Null Counts per column
for col in df.columns:
    null_count = int(df[col].isnull().sum()) # Explicitly convert to standard Python int
    if null_count > 0:
        data_quality_report['before_cleaning']['issues']['null_counts'][col] = null_count

# 2. Duplicate Row Count
duplicate_rows = int(df.duplicated().sum()) # Explicitly convert to standard Python int
data_quality_report['before_cleaning']['issues']['duplicate_rows'] = duplicate_rows

# 3. Type Mismatches (initial assessment based on inferred dtypes)
# For this dataset, pandas inferred types are mostly correct.
# More specific checks would require domain knowledge or a predefined schema.
# We will record the current dtypes as a baseline.
inferred_dtypes = {col: str(df[col].dtype) for col in df.columns}
data_quality_report['before_cleaning']['issues']['type_mismatches']['inferred_dtypes'] = inferred_dtypes

# Display the initial data quality report
print("Initial Data Quality Report (Before Cleaning):")
import json
print(json.dumps(data_quality_report['before_cleaning'], indent=2))

Initial Data Quality Report (Before Cleaning):
{
  "row_count": 100,
  "issues": {
    "null_counts": {},
    "duplicate_rows": 0,
    "type_mismatches": {
      "inferred_dtypes": {
        "userId": "int64",
        "id": "int64",
        "title": "object",
        "body": "object"
      }
    },
    "out_of_range_values": {},
    "inconsistent_string_formats": {}
  }
}


In [3]:
df_clean = df.copy()

# 4. Apply transformations and enrichments
# a. Word count for 'body' and 'title'
df_clean['body_word_count'] = df_clean['body'].apply(lambda x: len(str(x).split()))
df_clean['title_word_count'] = df_clean['title'].apply(lambda x: len(str(x).split()))

# b. Title casing for 'title'
original_titles = df_clean['title'].copy()
df_clean['title'] = df_clean['title'].apply(lambda x: str(x).title())

# Track changes for the audit report (specifically title casing inconsistencies)
# Assuming a title should always be title-cased. If not, it's an inconsistency.
title_inconsistencies = int((original_titles != df_clean['title']).sum()) # Explicitly convert to int
data_quality_report['before_cleaning']['issues']['inconsistent_string_formats']['title_casing'] = title_inconsistencies

# Display the transformed DataFrame head and info
print("\nDataFrame after transformations (head):")
display(df_clean.head())
print("\nDataFrame after transformations (info):")
df_clean.info()


DataFrame after transformations (head):


,userId,id,title,body,body_word_count,title_word_count
0,1,1,Sunt Aut Facere Repellat Provident Occaecati E...,quia et suscipit\nsuscipit recusandae consequu...,23,9
1,1,2,Qui Est Esse,est rerum tempore vitae\nsequi sint nihil repr...,31,3
2,1,3,Ea Molestias Quasi Exercitationem Repellat Qui...,et iusto sed quo iure\nvoluptatem occaecati om...,26,9
3,1,4,Eum Et Est Occaecati,ullam et saepe reiciendis voluptatem adipisci\...,28,4
4,1,5,Nesciunt Quas Odio,repudiandae veniam quaerat sunt sed\nalias aut...,23,3



DataFrame after transformations (info):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   userId            100 non-null    int64 
 1   id                100 non-null    int64 
 2   title             100 non-null    object
 3   body              100 non-null    object
 4   body_word_count   100 non-null    int64 
 5   title_word_count  100 non-null    int64 
dtypes: int64(4), object(2)
memory usage: 4.8+ KB


In [4]:
# c. Filtering (e.g., removing posts with userId > 5)
initial_rows = len(df_clean)
df_clean = df_clean[df_clean['userId'] <= 5]
rows_removed_by_filtering = int(initial_rows - len(df_clean)) # Explicitly convert to int

# d. Ranking based on 'body_word_count'
df_clean['body_word_count_rank'] = df_clean['body_word_count'].rank(method='dense', ascending=False)

# Update data quality report after cleaning
data_quality_report['after_cleaning']['row_count'] = len(df_clean)
data_quality_report['after_cleaning']['issues_fixed']['rows_removed_by_filtering'] = rows_removed_by_filtering
# The number of title casing inconsistencies fixed is directly the count we found before
data_quality_report['after_cleaning']['issues_fixed']['inconsistent_string_formats_fixed'] = data_quality_report['before_cleaning']['issues']['inconsistent_string_formats'].get('title_casing', 0)


print("\nDataFrame after filtering and ranking (head):")
display(df_clean.head())
print("\nDataFrame after filtering and ranking (info):")
df_clean.info()

print("\nUpdated Data Quality Report (After Cleaning):")
import json
print(json.dumps(data_quality_report, indent=2))


DataFrame after filtering and ranking (head):


,userId,id,title,body,body_word_count,title_word_count,body_word_count_rank
0,1,1,Sunt Aut Facere Repellat Provident Occaecati E...,quia et suscipit\nsuscipit recusandae consequu...,23,9,10.0
1,1,2,Qui Est Esse,est rerum tempore vitae\nsequi sint nihil repr...,31,3,2.0
2,1,3,Ea Molestias Quasi Exercitationem Repellat Qui...,et iusto sed quo iure\nvoluptatem occaecati om...,26,9,7.0
3,1,4,Eum Et Est Occaecati,ullam et saepe reiciendis voluptatem adipisci\...,28,4,5.0
4,1,5,Nesciunt Quas Odio,repudiandae veniam quaerat sunt sed\nalias aut...,23,3,10.0



DataFrame after filtering and ranking (info):
<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   userId                50 non-null     int64  
 1   id                    50 non-null     int64  
 2   title                 50 non-null     object 
 3   body                  50 non-null     object 
 4   body_word_count       50 non-null     int64  
 5   title_word_count      50 non-null     int64  
 6   body_word_count_rank  50 non-null     float64
dtypes: float64(1), int64(4), object(2)
memory usage: 3.1+ KB

Updated Data Quality Report (After Cleaning):
{
  "before_cleaning": {
    "row_count": 100,
    "issues": {
      "null_counts": {},
      "duplicate_rows": 0,
      "type_mismatches": {
        "inferred_dtypes": {
          "userId": "int64",
          "id": "int64",
          "title": "object",
          "body": "object"
       

In [5]:
import pandas as pd

# Prepare the audit report for CSV output
audit_report_data = []

# Before Cleaning
before_cleaning = data_quality_report['before_cleaning']
audit_report_data.append({
    'Metric': 'Row Count (Before Cleaning)',
    'Value': before_cleaning['row_count'],
    'Description': 'Number of rows in the DataFrame before any cleaning or transformations.'
})

# Handle quantifiable issues from 'before_cleaning'
if before_cleaning['issues']['null_counts']:
    for col, count in before_cleaning['issues']['null_counts'].items():
        audit_report_data.append({
            'Metric': f'Nulls in {col}',
            'Value': count,
            'Description': f'Number of null values found in column {col} before cleaning.'
        })

if before_cleaning['issues']['duplicate_rows'] > 0:
    audit_report_data.append({
        'Metric': 'Duplicate Rows (Before Cleaning)',
        'Value': before_cleaning['issues']['duplicate_rows'],
        'Description': 'Number of duplicate rows found before cleaning.'
    })

# Only add specific inconsistent string formats if they were detected and are quantifiable
if 'title_casing' in before_cleaning['issues']['inconsistent_string_formats'] and \
   before_cleaning['issues']['inconsistent_string_formats']['title_casing'] > 0:
    audit_report_data.append({
        'Metric': 'Title Casing Inconsistencies (Before Cleaning)',
        'Value': before_cleaning['issues']['inconsistent_string_formats']['title_casing'],
        'Description': 'Number of titles not in proper title case before cleaning.'
    })

# After Cleaning
after_cleaning = data_quality_report['after_cleaning']
audit_report_data.append({
    'Metric': 'Row Count (After Cleaning)',
    'Value': after_cleaning['row_count'],
    'Description': 'Number of rows in the DataFrame after cleaning and transformations.'
})

# Handle fixed issues from 'after_cleaning'
for issue_type, value in after_cleaning['issues_fixed'].items():
    if value > 0:
        audit_report_data.append({
            'Metric': f'{issue_type.replace("_", " ").title()}',
            'Value': value,
            'Description': f'Number of {issue_type.replace("_", " ")} fixed during cleaning.'
        })

# Calculate total issues existed and total fixed for the summary
# Ensure only quantifiable issues are summed
total_issues_existed = sum(before_cleaning['issues']['null_counts'].values()) + \
                       before_cleaning['issues']['duplicate_rows'] + \
                       before_cleaning['issues']['inconsistent_string_formats'].get('title_casing', 0)

total_issues_fixed = sum(after_cleaning['issues_fixed'].values())

audit_report_data.append({
    'Metric': 'Total Issues Existed (Before Cleaning)',
    'Value': total_issues_existed,
    'Description': 'Sum of all detected quantifiable issues before cleaning (nulls, duplicates, string formats).'
})
audit_report_data.append({
    'Metric': 'Total Issues Fixed (After Cleaning)',
    'Value': total_issues_fixed,
    'Description': 'Sum of all issues addressed or rows removed during cleaning.'
})


audit_df = pd.DataFrame(audit_report_data)

# Save to CSV
audit_report_filename = 'data_quality_audit_report.csv'
audit_df.to_csv(audit_report_filename, index=False)

print(f"\nData Quality Audit Report saved to {audit_report_filename}")

# Display the audit report as a formatted table
print("\n--- Data Quality Audit Report ---")
display(audit_df)


Data Quality Audit Report saved to data_quality_audit_report.csv

--- Data Quality Audit Report ---


,Metric,Value,Description
0,Row Count (Before Cleaning),100,Number of rows in the DataFrame before any cle...
1,Title Casing Inconsistencies (Before Cleaning),100,Number of titles not in proper title case befo...
2,Row Count (After Cleaning),50,Number of rows in the DataFrame after cleaning...
3,Inconsistent String Formats Fixed,100,Number of inconsistent string formats fixed fi...
4,Rows Removed By Filtering,50,Number of rows removed by filtering fixed duri...
5,Total Issues Existed (Before Cleaning),100,Sum of all detected quantifiable issues before...
6,Total Issues Fixed (After Cleaning),150,Sum of all issues addressed or rows removed du...


In [10]:
# Install PyMySQL if not already installed
import os
import dotenv
from sqlalchemy import create_engine
import pymysql
from urllib.parse import quote_plus

dotenv.load_dotenv()

# MySQL connection details
# IMPORTANT: Replace with your actual database credentials and server details
DB_USER = 'root' # <--- REPLACE THIS
DB_PASSWORD = quote_plus(os.getenv("password"))
DB_HOST = 'localhost'  # e.g., 'localhost', '127.0.0.1', or a remote host # <--- REPLACE THIS
DB_PORT = 3306         # Default MySQL port
DB_NAME = 'task_2_db' # <--- REPLACE THIS
TABLE_NAME = 'cleaned_posts'

# Create a SQLAlchemy engine for MySQL
# Ensure you have 'PyMySQL' or 'mysql-connector-python' installed:
# pip install PyMySQL
# or
# pip install mysql-connector-python
try:
    engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

    # Load the cleaned DataFrame into MySQL
    # if_exists='replace' will drop the table and recreate it.
    # if_exists='append' will add rows to an existing table.
    # if_exists='fail' will raise an error if table exists.
    df_clean.to_sql(TABLE_NAME, engine, if_exists='replace', index=False)
    print(f"\nSuccessfully loaded cleaned data into MySQL table '{TABLE_NAME}' in database '{DB_NAME}'.")

except Exception as e:
    print(f"Error loading data to MySQL: {e}")
    print("Please ensure MySQL is running, your credentials are correct, and the database exists.")
    print("You might also need to install the MySQL connector: `pip install PyMySQL`")


Successfully loaded cleaned data into MySQL table 'cleaned_posts' in database 'task_2_db'.
